# Problem 4: Practical Implications & Real-World Utility Insights

**Objective:**
This notebook addresses the three critical practical questions in **Section 4 of the assignment**:
1. **Practical Energy-Saving / Customer-Insight Opportunity:** Designing an Eco-Wash & Peak-Shifting program, simulating energy/cost savings, and formulating an A/B testing measurement framework.
2. **Impact of 15-Minute and 30-Minute Smart-Meter Data:** Numerically simulating and plotting how lower sampling intervals dilute high-power transient signatures and compromise disaggregation.
3. **Domain Transfer & Limitations (UK to India):** Analyzing the structural and electrical hurdles of transferring UK-trained NILM models to Indian residential energy environments.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('figures', exist_ok=True)
print('Libraries imported successfully!')

## 1. Practical Energy-Saving Program (Peak Shift + Eco-Wash)
Based on our findings in Problem 2:
- Washing machines peak between **18:00 and 20:00** (highest grid carbon intensity and peak Time-of-Use tariffs).
- Intensive hot cycles (House 2) consume **0.613 kWh**, while 30°C eco-wash cycles (House 1) consume only **0.309 kWh**.

We simulate the impact of a utility behavioral nudge program:

In [ ]:
cycles_per_year = 4 * 52  # 4 cycles/week
annual_hot_kwh = cycles_per_year * 0.613
annual_eco_kwh = cycles_per_year * 0.309
kwh_saved_per_home = annual_hot_kwh - annual_eco_kwh

# Economy 7 Time-of-Use Tariffs
cost_baseline = annual_hot_kwh * 0.35   # 35p/kWh peak
cost_optimized = annual_eco_kwh * 0.12  # 12p/kWh off-peak
annual_savings_per_home = cost_baseline - cost_optimized

# Utility Pilot Cohort (50,000 households)
cohort_size = 50000
total_grid_mwh = (kwh_saved_per_home * cohort_size) / 1000.0
total_savings_gbp = annual_savings_per_home * cohort_size

print(f'Annual Energy Saved per Home:   {kwh_saved_per_home:.1f} kWh/year ({kwh_saved_per_home/annual_hot_kwh:.1%})')
print(f'Annual Bill Savings per Home:  £{annual_savings_per_home:.2f}/year')
print(f'Pilot Scale (50,000 Homes):     {total_grid_mwh:,.1f} MWh/year saved (£{total_savings_gbp:,.2f})')

### Experimental A/B Testing Measurement Framework
To prove the program works without confounding factors:
1. **Randomized Cohort:** Assign 5,000 smart-meter customers to Treatment (app nudges) and 5,000 to Control.
2. **Primary Metric:** Percentage of laundry cycles run during peak hours (17:00–21:00) before vs. after intervention.
3. **Secondary Metric:** Mean kWh per detected cycle (measures adoption of 30°C eco-wash settings).
4. **Statistical Rigor:** Evaluate with a **Difference-in-Differences (DiD)** regression model to control for seasonal temperature shifts.

## 2. Technical Impact of 15-Minute and 30-Minute Smart-Meter Data
Standard commercial smart meters often report at 15-minute or 30-minute intervals.
Here, we take ground-truth House 1 data and downsample it to 15-min and 30-min resolution to measure signal degradation:

In [ ]:
DATA_PATH = '../house1_cleaned_1min.parquet'
df = pd.read_parquet(DATA_PATH)

# 4-hour morning window on October 23, 2013
sample_window = df.loc['2013-10-23 05:00:00':'2013-10-23 09:00:00'].copy()

sample_1min = sample_window['Appliance5']
sample_15min = sample_1min.resample('15min').mean()
sample_30min = sample_1min.resample('30min').mean()

agg_1min = sample_window['Aggregate']
agg_15min = agg_1min.resample('15min').mean()
agg_30min = agg_1min.resample('30min').mean()

print(f'1-Minute Peak:  {sample_1min.max():.1f} W')
print(f'15-Minute Peak: {sample_15min.max():.1f} W (-{1 - sample_15min.max()/sample_1min.max():.1%})')
print(f'30-Minute Peak: {sample_30min.max():.1f} W (-{1 - sample_30min.max()/sample_1min.max():.1%})')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# 1-Min
axes[0].plot(sample_window.index, agg_1min, label='Aggregate Mains Power', color='gray', alpha=0.5, lw=1)
axes[0].plot(sample_window.index, sample_1min, label='Washing Machine (1-Min Resolution)', color='#d62728', lw=1.8)
axes[0].set_title('A. 1-Minute Resolution: Sharp 2,368 W Heating Spikes & Precise Cycle Boundaries', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Power (Watts)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right')

# 15-Min
axes[1].step(sample_15min.index, agg_15min, where='post', label='Aggregate Mains (15-Min)', color='gray', alpha=0.5, lw=1)
axes[1].step(sample_15min.index, sample_15min, where='post', label='Washing Machine (15-Min)', color='#ff7f0e', lw=2)
axes[1].set_title(f'B. 15-Minute Resolution: Peak Flattened to {sample_15min.max():.0f} W (-63% Dilution), Only 2-3 Samples per Cycle', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Power (Watts)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='upper right')

# 30-Min
axes[2].step(sample_30min.index, agg_30min, where='post', label='Aggregate Mains (30-Min)', color='gray', alpha=0.5, lw=1)
axes[2].step(sample_30min.index, sample_30min, where='post', label='Washing Machine (30-Min)', color='#2ca02c', lw=2)
axes[2].set_title(f'C. 30-Minute Resolution: Peak Flattened to {sample_30min.max():.0f} W (-79% Dilution), Cycle Blends into Baseload', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Power (Watts)')
axes[2].set_xlabel('Time of Day (October 23, 2013)')
axes[2].grid(True, linestyle='--', alpha=0.5)
axes[2].legend(loc='upper right')

plt.suptitle('Impact of Smart-Meter Data Resolution on Appliance Disaggregation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/problem4_sampling_resolution_impact.png', dpi=200)
plt.show()

## 3. Domain Transfer to India & Operational Limitations

### Why UK NILM Models Fail When Directly Transferred to India:
1. **Absence of Internal Water Heating:**
   - **UK Homes:** Almost all washing machines are front-loading units equipped with an internal 2,000–2,500 W electric heating element.
   - **Indian Homes:** The vast majority of domestic machines are top-load or semi-automatic cold-water fill. Motor agitation draws only **200–500 Watts** with no high-power resistive heating step.
   - **Impact:** A UK-trained model that searches for 2 kW heating steps will completely fail to detect active laundry in typical Indian households.
2. **Alternative High-Power Loads in Indian Grids:**
   - Indian homes feature dedicated water geysers (2–3 kW), room air conditioners (1.5–2.5 kW), and induction cooktops. These devices create high-power square pulses that a UK-trained model would mistakenly classify as washing machine heaters.
3. **Overlapping Signatures:**
   - Active power alone cannot separate purely resistive appliances (kettles, toasters, space heaters) from washing machine heating elements without reactive power (VAR) or high-frequency transient sampling.